# IndieFake (Indian-accent) SSL training pipeline -- Colab

Picks up the `ai_voice_detector` pipeline at step 4 (SSL feature extraction), which was too slow on an 8GB CPU-only machine. Uses Colab's GPU (`Runtime > Change runtime type > T4 GPU`) instead.

**Before running**:
- Put the 4 IndieFake zip parts (`drive-download-...-00{1..4}.zip`) somewhere in your Google Drive and set `INDIEFAKE_ZIP_DIR` below to that folder.
- Upload `asvspoof_realworld_cache.zip` (pre-built `data/`, `data_realworld/`, `cache/ssl_embeddings/` -- see the markdown cell before the restore step for why this replaced re-fetching) to your Drive and set `CACHE_ZIP_PATH` below to it.

Steps: mount Drive -> clone repo -> restore ASVspoof + real-world audio from a pre-built Drive zip (datashare.ed.ac.uk's WAF blocks re-fetching from some Colab/GCP IPs) -> run the Indian dataset pipeline (organize -> verify -> degrade -> manifests -> extract embeddings on GPU) -> train -> evaluate (5-quadrant + leave-one-generator-out) -> push results back.

In [28]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
# EDIT THIS to wherever the 4 IndieFake zip parts live in your Drive
INDIEFAKE_ZIP_DIR = "/content/drive/MyDrive/IndieFake  Dataset"

In [4]:
REPO_URL = "https://github.com/Jeevan-Cyber-Sai/sih.git"
!git clone $REPO_URL /content/sih
%cd /content/sih/ai_voice_detector

Cloning into '/content/sih'...
remote: Enumerating objects: 6685, done.
remote: Total 6685 (delta 0), reused 0 (delta 0), pack-reused 6685 (from 1)
Receiving objects: 100% (6685/6685), 783.03 MiB | 33.69 MiB/s, done.
Resolving deltas: 100% (50/50), done.
Updating files: 100% (11681/11681), done.
/content/sih/ai_voice_detector


In [5]:
!pip install -q -r requirements.txt

In [6]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- set Runtime > Change runtime type > T4 GPU')

CUDA available: True
device: Tesla T4


## Restore ASVspoof + real-world audio (from Drive, not re-fetched)

`data/`, `data_realworld/` audio isn't in git (gitignored, large binary data). Re-fetching them from Colab used to work here (`download_asvspoof_subset.py` hits `datashare.ed.ac.uk`), but that host's WAF now hard-blocks some GCP egress IPs with a 403 "suspected bot" page. It's IP/ASN-based, not header-based -- confirmed by hitting the same URL with the same browser User-Agent from a non-GCP IP and getting a normal `206`, so no amount of header-tweaking fixes it. It's also intermittent (depends on which Colab backend VM you land on, not a blanket block), so retrying isn't reliable either.

Instead of gambling on a retry, this restores `data/`, `data_realworld/`, and `cache/ssl_embeddings/` from a zip built locally with the same deterministic download (same seeds, same public sources) -- byte-identical to what a successful re-fetch here would produce. It also skips re-extracting SSL embeddings for the base datasets on GPU, since the cache is portable (keyed by relative path).

**Before running the next cell**: upload `asvspoof_realworld_cache.zip` to your Drive and set `CACHE_ZIP_PATH` below to it.</cell_type>

In [ ]:
CACHE_ZIP_PATH = "/content/drive/MyDrive/IndieFake_Datasets/asvspoof_realworld_cache.zip"  # EDIT to wherever you uploaded it
!unzip -oq "$CACHE_ZIP_PATH" -d /content/sih/ai_voice_detector
print('restored data/, data_realworld/, cache/ssl_embeddings/ from Drive')

In [20]:
!cd /content/sih && git pull


Already up to date.


In [22]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f.lower().endswith('.zip'):
            print(os.path.join(root, f))

/content/drive/MyDrive/Notes.zip


## Indian dataset pipeline (steps 1-3, replayed here since they're cheap)

In [12]:
import os
os.environ['INDIEFAKE_ZIP_DIR'] = INDIEFAKE_ZIP_DIR
!python scripts/organize_indian_dataset.py

extracting drive-download-20260903T071406Z-1-001.zip ...
Traceback (most recent call last):
  File "/content/sih/ai_voice_detector/scripts/organize_indian_dataset.py", line 85, in <module>
    main()
    ~~~~^^
  File "/content/sih/ai_voice_detector/scripts/organize_indian_dataset.py", line 47, in main
    with zipfile.ZipFile(zip_path) as zf:
         ~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/usr/lib/python3.13/zipfile/__init__.py", line 1389, in __init__
    self.fp = io.open(file, filemode)
              ~~~~~~~^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/IndieFake  Dataset/drive-download-20260903T071406Z-1-001.zip'


In [ ]:
!python verify_indian_dataset.py

In [ ]:
!python scripts/degrade_indian_dataset.py

In [ ]:
# Idempotent -- these will report 0 new entries since the manifests are
# already committed to git and cloned above. Harmless to (re-)run.
!python scripts/build_holdout_manifest_indian.py
!python scripts/build_generator_manifest_indian.py

## Step 4: extract SSL embeddings on GPU

`features_ssl.py` now auto-detects CUDA and moves the model/inputs there. `MAX_WORKERS=1` in the script is intentional even here -- one process already saturates a single GPU; more workers would just fight over it via separate CUDA contexts.

In [ ]:
!python scripts/extract_indian_ssl_features.py

## Step 5: train the combined classifier

In [ ]:
!python train_ssl.py --indian

## Step 6: six-bucket evaluation (clean / real-world / Indian x real / fake)

In [ ]:
!python scripts/five_quadrant_eval.py

## Step 7: leave-one-generator-out, including the new 'indiefake' generator

In [ ]:
!python scripts/leave_one_generator_out_eval.py

## Bring results back

`models/*.joblib` and `features_indian.npy` are gitignored (large/regenerable) -- copy them to Drive to download. The new Indian embeddings in `cache/ssl_embeddings/` ARE meant to be committed (same convention as the existing ASVspoof/real-world cache) so any machine that later `git pull`s gets them for free -- this needs a GitHub token with push access to this repo, entered interactively below (never hardcode it in the notebook).

In [ ]:
!mkdir -p /content/drive/MyDrive/sih_indian_results
!cp models/voice_classifier_ssl_indian.joblib models/scaler_ssl_indian.joblib features_indian.npy /content/drive/MyDrive/sih_indian_results/
print('copied to Drive: sih_indian_results/')

In [ ]:
import getpass
token = getpass.getpass('GitHub personal access token (repo scope, push rights): ')
!git config user.email "jeevanarhack@gmail.com"
!git config user.name "Jeevan Sai V"
!git add cache/ssl_embeddings data_realworld/holdout_manifest.json data_realworld/generator_manifest.json
!git commit -m "Add Indian-accent SSL embeddings from Colab run"
!git push https://{token}@github.com/Jeevan-Cyber-Sai/sih.git HEAD:main